In [0]:
# %pip install -r https://raw.githubusercontent.com/seaninc-training/databricks-ai-project/refs/heads/main/requirements.txt

In [0]:
# Imports and Variable Set Up
import logging
import json
from databricks.vector_search.client import VectorSearchClient
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Catalog / Schema / Volume
catalog = "workspace"
schema = "ai_project"

# AI Gateway / Model config
base_url = "https://7474648118426063.ai-gateway.cloud.databricks.com/mlflow/v1"
llm_model = "databricks-meta-llama-3-1-405b-instruct"

# Client Setup
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=base_url
)

# Vector Search config
vsc = VectorSearchClient()
endpoint_name = "search_endpoint"
book_index_name = f"{catalog}.{schema}.book_vector_index"
doc_index_name = f"{catalog}.{schema}.doc_vector_index"

# File config
source_config = {
    "books": {
        "index": book_index_name
    },
    "docs": {
        "index": doc_index_name
    }
}

In [0]:
%run ../utils/logging_utils

In [0]:
%run ../utils/search_utils

In [0]:
%run ../utils/agent_tools

In [0]:
# Diagnostic
# ⚙️ Configure before running

# -> Book Test
# run_diagnostic(
#     source_type="books",
#     query="What was the weapon Raskolnikov used in the crime?"
# )

# -> Doc Test
run_diagnostic(
    source_type="docs",
    query="How do I create a managed table in Databricks?"
)

In [0]:
# Cell 8: Agent Setup

# Initialize Chat History
if 'chat_history' not in globals():
    chat_history = []
    logger.info("🧠 Memory Initialized.")

# Tool Schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description": "Search the vector index for relevant context from books or technical documentation. Always call this before execute_in_databricks to retrieve grounded context for code generation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "The search query."
                    },
                    "source_type": {
                        "type": "string",
                        "enum": ["books", "docs"],
                        "description": "The source domain to search. Use 'books' for literary content and 'docs' for technical documentation."
                    }
                },
                "required": ["question", "source_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "execute_in_databricks",
            "description": "Presents proposed SQL code to the user for review and executes it upon confirmation. Always generate SQL grounded in retrieved documentation before calling this tool. Requires explicit user confirmation before execution.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql_code": {
                        "type": "string",
                        "description": "The SQL code to present to the user for review and potential execution."
                    }
                },
                "required": ["sql_code"]
            }
        }
    }
]

In [0]:
def clean_message(message):
    dumped = message.model_dump()
    # Only keep fields supported by Databricks AI Gateway
    allowed_keys = {"role", "content", "tool_calls", "tool_call_id", "name"}
    return {k: v for k, v in dumped.items() if k in allowed_keys and v is not None}

In [0]:
def run_agent(user_query):
    global chat_history

    # 1. System Prompt
    system_prompt = {
        "role": "system",
        "content": """You are an intelligent Data Engineering Assistant with access to technical documentation and literary sources.

        RULES:
        1. Always use the search_documents tool to retrieve context before answering.
        2. Choose source_type carefully:
           - Use 'docs' for technical questions about Databricks, SQL, or data engineering concepts
           - Use 'books' for questions about literary content
        3. Only use retrieved context to form your answer. If nothing relevant is found, state that clearly.
        4. Always cite the Source Page and File Name from retrieved context.
        5. If a definitive answer isn't in the retrieved data, explain what IS there and clarify the uncertainty.
        6. When execute_in_databricks returns a success or cancellation message, summarize the outcome to the user. Do not ask for confirmation again — execution has already been handled."""
    }

    # 2. Build conversation thread
    messages = [system_prompt] + chat_history + [{"role": "user", "content": user_query}]

    # 3. Initial Call (Thinking Phase)
    response = client.chat.completions.create(
        model=llm_model,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    # Cost Tracker (Part 1)
    total_tokens = response.usage.total_tokens
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # 4. Agentic Tool Execution Loop
    if tool_calls:
        messages.append(clean_message(response_message))

        while tool_calls:
            execution_happened = False

            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                query_args = json.loads(tool_call.function.arguments)

                if tool_name == "search_documents":
                    observations = search_documents(
                        query=query_args['question'],
                        source_type=query_args['source_type']
                    )

                    if observations:
                        context_blocks = [
                            f"Source Page {row[2]} (File: {row[1]}, Index: {row[3]}): {row[0]}"
                            for row in observations
                        ]
                        context = "\n---\n".join(context_blocks)
                    else:
                        context = "No relevant results found in the index."

                    logger.info(f"📡 Retrieved {len(observations)} excerpts from '{query_args['source_type']}' index.")

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": "search_documents",
                        "content": context
                    })

                elif tool_name == "execute_in_databricks":
                    execution_result = execute_in_databricks(
                        sql_code=query_args['sql_code']
                    )

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": "execute_in_databricks",
                        "content": execution_result
                    })

                    execution_happened = True

            # Check if LLM wants to call another tool
            next_response = client.chat.completions.create(
                model=llm_model,
                messages=messages,
                tools=tools,
                tool_choice="none" if execution_happened else "auto"
            )

            total_tokens += next_response.usage.total_tokens
            next_message = next_response.choices[0].message
            tool_calls = next_message.tool_calls

            if tool_calls:
                # LLM wants to call another tool — continue loop
                messages.append(clean_message(response_message))

            else:
                # LLM is done with tools — this is the final answer
                answer = next_message.content

    else:
        answer = response_message.content

    # 5. Update Memory
    chat_history.append({"role": "user", "content": user_query})
    chat_history.append({"role": "assistant", "content": answer})

    # 6. Display Response
    print(f"\n{'='*50}")
    print(f"🤖 AGENT RESPONSE:\n{answer}")
    print(f"{'='*50}")
    print(f"💰 USAGE: {total_tokens} tokens")

    return answer

In [0]:
# Cell 10: Run Agent
# run_agent("What weapon did Raskolnikov use to commit the crime?")
# run_agent("How do I create a managed table in Databricks?")
# run_agent("What are some best practices for structuring a Databricks notebook?")

In [0]:
# run_agent("Create a table called workspace.ai_project.test_table_3 with columns id_3 INT and name_3 STRING")